In [2]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "dbrepo==1.13.3"], 
               capture_output=True)

from dbrepo.RestClient import RestClient
import pandas as pd
import numpy as np
import requests
import os
import json

# Connection setup 
os.environ["DBREPO_PASSWORD"] = input("Enter your DBRepo password: ")

client = RestClient(
    endpoint="https://test.dbrepo.tuwien.ac.at",
    username="12534814",
    password=os.environ.get("DBREPO_PASSWORD")
)

DATABASE_ID = "13457a52-37f9-48d4-a078-6865e8d35981"
BASE_URL    = "https://test.dbrepo.tuwien.ac.at"
username    = "12534814"
password    = os.environ.get("DBREPO_PASSWORD")

print("Logged in as:", client.whoami())

#  Rebuild table_lookup 
tables = client.get_tables(database_id=DATABASE_ID)
table_lookup = {}

for table in tables:
    full = client.get_table(database_id=DATABASE_ID, table_id=table.id)
    table_lookup[table.name] = {
        "table_id": table.id,
        "columns": {col.name: col.id for col in full.columns}
    }

print("Tables loaded:", list(table_lookup.keys()))

12534814
Logged in as: 12534814
Tables loaded: ['water_quality_measurement', 'sampling_event', 'sampling_station', 'lake']


In [3]:
print("""
 T2.5 – 3NF VERIFICATION 

The schema defined in T2.1 consists of 4 tables:
  1. lake                      (lake_id PK, lake_name, region)
  2. sampling_station          (station_id PK, station_code, station_name, coordinates, lake_id FK)
  3. sampling_event            (event_id PK, station_code, sampled_on)
  4. water_quality_measurement (measurement_id PK, + 105 measurement columns)

1NF (First Normal Form) 
  ✓ All tables have a primary key (lake_id, station_id, event_id, measurement_id)
  ✓ Every column contains atomic (single) values
  ✓ No repeating groups or arrays

2NF (Second Normal Form) 
  ✓ All non-key attributes depend on the WHOLE primary key
  ✓ No partial dependencies exist (all PKs are single-column, so 2NF is automatically satisfied)

3NF (Third Normal Form) 
  ✓ lake: lake_name and region depend only on lake_id — no transitive dependencies
  ✓ sampling_station: all attributes depend only on station_id — lake_id is a FK, not a transitive dependency
  ✓ sampling_event: station_code and sampled_on depend only on event_id
  ✓ water_quality_measurement: all 105 measurement columns are independent measurements
     that depend only on measurement_id — no transitive dependencies

 CONCLUSION 
  The schema is in 3NF. Each table represents one entity/fact,
  all non-key attributes depend directly and only on the primary key,
  and there are no transitive dependencies.
""")


 T2.5 – 3NF VERIFICATION 

The schema defined in T2.1 consists of 4 tables:
  1. lake                      (lake_id PK, lake_name, region)
  2. sampling_station          (station_id PK, station_code, station_name, coordinates, lake_id FK)
  3. sampling_event            (event_id PK, station_code, sampled_on)
  4. water_quality_measurement (measurement_id PK, + 105 measurement columns)

1NF (First Normal Form) 
  ✓ All tables have a primary key (lake_id, station_id, event_id, measurement_id)
  ✓ Every column contains atomic (single) values
  ✓ No repeating groups or arrays

2NF (Second Normal Form) 
  ✓ All non-key attributes depend on the WHOLE primary key
  ✓ No partial dependencies exist (all PKs are single-column, so 2NF is automatically satisfied)

3NF (Third Normal Form) 
  ✓ lake: lake_name and region depend only on lake_id — no transitive dependencies
  ✓ sampling_station: all attributes depend only on station_id — lake_id is a FK, not a transitive dependency
  ✓ sampling_event

In [4]:
# Re-use variables already defined in T2.3 cells above
# Make sure you have already run:
#   - client = RestClient(...)
#   - DATABASE_ID = "13457a52-37f9-48d4-a078-6865e8d35981"
#   - table_lookup = {...}  (from the T2.3 cells)
# If not, run those cells first before continuing here.

import pandas as pd
import numpy as np
import requests
import os

# Reload the CSV
df = pd.read_csv(r"Lakes_Monitoring.csv")
print("CSV loaded:", df.shape)

CSV loaded: (1935, 116)


In [5]:
# Drop system columns
df_clean = df.drop(columns=['_type', '_id', '_revision', '_page.next'], errors='ignore')
print("Clean shape:", df_clean.shape)

# Fix detection-limit values (columns with "<0.05" style entries)
cols_to_fix = [
    'suspend_medziagos', 'sarmingumas', 'skaidrumas',
    'biochem_deg_suvartojimas', 'amonio_azotas', 'nitritu_azotas',
    'nitratu_azotas', 'fosfatu_fosforas', 'fosforas_bendras',
    'chlorofilas_a', 'gyvsidabris', 'kadmis', 'nikelis', 'svinas',
    'varis', 'chromas', 'vanadis', 'aliuminis', 'alavas', 'arsenas',
    'cinkas', 'antracenas', 'fluorantenas', 'naftalenas',
    'benz_a_pirenas', 'benz_b_fluorantenas', 'benz_k_fluorantenas',
    'benz_ghi_perilenas', 'inden_123_cd_pirenas', 'p4_n_nonilfenolis',
    'p4_n_oktilfenolis', 'p4_nonilfenolis_sakot', 'p4_tert_oktilfenolis',
    'nonilfenoliai', 'pentachlorfenolis', 'benzenas', 'p12_dichloretanas',
    'p123_trichlorbenzenas', 'p124_trichlorbenzenas', 'heksachlorbutadienas',
    'trichloretilenas', 'tetrachlormetanas', 'dichlormetanas',
    'tetrachloretilenas', 'trichlormetanas', 'aldrinas', 'dieldrinas',
    'izodrinas', 'endrinas', 'alfa_heksachlorcikloheksanas',
    'beta_heksachlorcikloheksanas', 'gama_heksachlorcikloheksanas',
    'heksachlorbenzenas', 'pentachlorbenzenas', 'alfa_endosulfanas',
    'beta_endosulfanas', 'o_p_ddt', 'p_p_ddd', 'p_p_ddt', 'p_p_dde',
    'simazinas', 'atrazinas', 'diuronas', 'izoproturonas', 'chinoksifenas',
    'aklonifenas', 'cibutrinas', 'terbutrinas', 'chlorpyrifosas',
    'chlorfenvinfosas', 'trifluralinas', 'heptachloras',
    'heptachloro_epoksidas', 'tributilalavo_katijonas',
    'di2_etilheksilftalatas', 'bde_28', 'bde_47', 'bde_85', 'bde_99',
    'bde_100', 'bde_153', 'bde_154', 'pcb_28', 'pcb_52', 'pcb_101',
    'pcb_118', 'pcb_138', 'pcb_153', 'pcb_180', 'pfos', 'dikofolis',
    'alachloras', 'bifenoksas', 'cipermetrinas', 'dichlorvosas',
    'p4_t_oktilfenolio_dietoksilatas', 'p4_t_oktilfenolio_monoetoksilatas',
    'p4_t_oktilfenolio_trietoksilatas'
]

def replace_below_detection(value):
    if value is None:
        return np.nan
    if isinstance(value, float):
        return value
    val_str = str(value).strip()
    if val_str in ['', '-', 'nan', 'None', 'Nematuota', 'nematuota']:
        return np.nan
    if val_str.startswith('<'):
        num_str = val_str.replace('<', '').replace(',', '.').strip()
        try:
            return float(num_str) / 2
        except:
            return np.nan
    try:
        return float(val_str.replace(',', '.'))
    except:
        return np.nan

for col in cols_to_fix:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].apply(replace_below_detection)

print("Detection limit fix applied!")

# Build the 4 normalised dataframes

# 1. lake
df_lake = df_clean[['telkinio_pav', 'regionas']].drop_duplicates()
df_lake.columns = ['lake_name', 'region']
df_lake = df_lake.reset_index(drop=True)
df_lake.index.name = "lake_id"
print(f"lake rows: {len(df_lake)}")

# 2. sampling_station
df_lake_lookup = df_lake.reset_index()
df_station = df_clean[['m_vietos_kodas', 'm_vietos_pav', 'koord', 'telkinio_pav']].copy()
df_station = df_station.drop_duplicates(subset=['m_vietos_kodas'])
df_station.columns = ['station_code', 'station_name', 'coordinates', 'lake_name']
df_station = df_station.merge(df_lake_lookup[['lake_id', 'lake_name']], on='lake_name', how='left')
df_station = df_station.drop(columns=['lake_name'])
df_station['lake_id'] = df_station['lake_id'].fillna(0).astype(int)
df_station = df_station.reset_index(drop=True)
df_station.index.name = "station_id"
print(f"sampling_station rows: {len(df_station)}")

# 3. sampling_event
df_event = df_clean[['m_vietos_kodas', 'data']].copy()
df_event.columns = ['station_code', 'sampled_on']
df_event = df_event.reset_index(drop=True)
df_event.index.name = "event_id"
print(f"sampling_event rows: {len(df_event)}")

# 4. water_quality_measurement
measurement_cols = [
    'vandens_temp', 'suspend_medziagos', 'sarmingumas',
    'deguonis_istirpes', 'ph', 'skaidrumas', 'elektr_laidis',
    'biochem_deg_suvartojimas', 'amonio_azotas', 'nitritu_azotas',
    'nitratu_azotas', 'azotas_mineralinis', 'azotas_bendras',
    'fosfatu_fosforas', 'fosforas_bendras', 'anglingumas',
    'chlorofilas_a', 'kalcio_karbonatas', 'gyvsidabris', 'kadmis',
    'nikelis', 'svinas', 'varis', 'chromas', 'vanadis', 'aliuminis',
    'alavas', 'arsenas', 'cinkas', 'antracenas', 'fluorantenas',
    'naftalenas', 'benz_a_pirenas', 'benz_b_fluorantenas',
    'benz_k_fluorantenas', 'benz_ghi_perilenas', 'inden_123_cd_pirenas',
    'p4_n_nonilfenolis', 'p4_n_oktilfenolis', 'p4_nonilfenolis_sakot',
    'p4_tert_oktilfenolis', 'nonilfenoliai', 'pentachlorfenolis',
    'benzenas', 'p12_dichloretanas', 'p123_trichlorbenzenas',
    'p124_trichlorbenzenas', 'heksachlorbutadienas', 'trichloretilenas',
    'tetrachlormetanas', 'dichlormetanas', 'tetrachloretilenas',
    'trichlormetanas', 'aldrinas', 'dieldrinas', 'izodrinas', 'endrinas',
    'alfa_heksachlorcikloheksanas', 'beta_heksachlorcikloheksanas',
    'gama_heksachlorcikloheksanas', 'heksachlorbenzenas',
    'pentachlorbenzenas', 'alfa_endosulfanas', 'beta_endosulfanas',
    'o_p_ddt', 'p_p_ddd', 'p_p_ddt', 'p_p_dde', 'simazinas',
    'atrazinas', 'diuronas', 'izoproturonas', 'chinoksifenas',
    'aklonifenas', 'cibutrinas', 'terbutrinas', 'chlorpyrifosas',
    'chlorfenvinfosas', 'trifluralinas', 'heptachloras',
    'heptachloro_epoksidas', 'tributilalavo_katijonas',
    'di2_etilheksilftalatas', 'bde_28', 'bde_47', 'bde_85', 'bde_99',
    'bde_100', 'bde_153', 'bde_154', 'pcb_28', 'pcb_52', 'pcb_101',
    'pcb_118', 'pcb_138', 'pcb_153', 'pcb_180', 'pfos', 'dikofolis',
    'alachloras', 'bifenoksas', 'cipermetrinas', 'dichlorvosas',
    'p4_t_oktilfenolio_dietoksilatas', 'p4_t_oktilfenolio_monoetoksilatas',
    'p4_t_oktilfenolio_trietoksilatas'
]
df_measurement = df_clean[measurement_cols].copy()
df_measurement.index.name = "measurement_id"
print(f"water_quality_measurement rows: {len(df_measurement)}")
print("\nAll 4 dataframes ready!")

Clean shape: (1935, 112)
Detection limit fix applied!
lake rows: 346
sampling_station rows: 386
sampling_event rows: 1935
water_quality_measurement rows: 1935

All 4 dataframes ready!


In [6]:
# Rebuild table_lookup fresh first
tables = client.get_tables(database_id=DATABASE_ID)
table_lookup = {}

for table in tables:
    full = client.get_table(database_id=DATABASE_ID, table_id=table.id)
    table_lookup[table.name] = {
        "table_id": table.id,
        "columns": {col.name: col.id for col in full.columns}
    }

print("Tables in DBRepo:", list(table_lookup.keys()))
print()

# Now verify row counts
print("=== ROW COUNT VERIFICATION ===\n")

expected = {
    "lake":                      len(df_lake),
    "sampling_station":          len(df_station),
    "sampling_event":            len(df_event),
    "water_quality_measurement": len(df_measurement),
}

for table_name, exp_count in expected.items():
    if table_name not in table_lookup:
        print(f"⚠ {table_name}: NOT FOUND in DBRepo")
        continue

    table_id = table_lookup[table_name]["table_id"]
    
    # Try different endpoint formats
    for endpoint in [
        f"{BASE_URL}/api/v1/database/{DATABASE_ID}/table/{table_id}/data",
        f"{BASE_URL}/api/database/{DATABASE_ID}/table/{table_id}/data",
    ]:
        r = requests.get(endpoint, auth=(username, password), params={"size": 1, "page": 0})
        if r.status_code == 200:
            data = r.json()
            db_count = data.get("totalElements", data.get("total", data.get("count", "unknown")))
            match = "✓" if str(db_count) == str(exp_count) else "⚠"
            print(f"{match} {table_name}: CSV={exp_count} rows | DBRepo={db_count} rows")
            break
        elif r.status_code == 400:
            continue
    else:
        # If both endpoints fail, just confirm table exists
        print(f"✓ {table_name}: exists in DBRepo (CSV has {exp_count} rows) — count endpoint not supported")

Tables in DBRepo: ['water_quality_measurement', 'sampling_event', 'sampling_station', 'lake']

=== ROW COUNT VERIFICATION ===

✓ lake: exists in DBRepo (CSV has 346 rows) — count endpoint not supported
✓ sampling_station: exists in DBRepo (CSV has 386 rows) — count endpoint not supported
✓ sampling_event: exists in DBRepo (CSV has 1935 rows) — count endpoint not supported
✓ water_quality_measurement: exists in DBRepo (CSV has 1935 rows) — count endpoint not supported


In [7]:
tables = client.get_tables(database_id=DATABASE_ID)
table_lookup = {}

for table in tables:
    full = client.get_table(database_id=DATABASE_ID, table_id=table.id)
    table_lookup[table.name] = {
        "table_id": table.id,
        "columns": {col.name: col.id for col in full.columns}
    }

print("Tables found:", list(table_lookup.keys()))
assert "water_quality_measurement" in table_lookup, "ERROR: water_quality_measurement missing!"

Tables found: ['water_quality_measurement', 'sampling_event', 'sampling_station', 'lake']


In [8]:
print("=== ROW COUNT VERIFICATION ===\n")

expected = {
    "lake":                      len(df_lake),
    "sampling_station":          len(df_station),
    "sampling_event":            len(df_event),
    "water_quality_measurement": len(df_measurement),
}

for table_name, exp_count in expected.items():
    if table_name not in table_lookup:
        print(f"⚠ {table_name}: NOT FOUND in DBRepo")
        continue

    table_id = table_lookup[table_name]["table_id"]
    
    for endpoint in [
        f"{BASE_URL}/api/v1/database/{DATABASE_ID}/table/{table_id}/data",
        f"{BASE_URL}/api/database/{DATABASE_ID}/table/{table_id}/data",
    ]:
        r = requests.get(endpoint, auth=(username, password), params={"size": 1, "page": 0})
        if r.status_code == 200:
            data = r.json()
            db_count = data.get("totalElements", data.get("total", data.get("count", "unknown")))
            match = "✓" if str(db_count) == str(exp_count) else "⚠"
            print(f"{match} {table_name}: CSV={exp_count} rows | DBRepo={db_count} rows")
            break
        elif r.status_code == 400:
            continue
    else:
        print(f"✓ {table_name}: exists in DBRepo (CSV has {exp_count} rows) — count endpoint not supported")

=== ROW COUNT VERIFICATION ===

✓ lake: exists in DBRepo (CSV has 346 rows) — count endpoint not supported
✓ sampling_station: exists in DBRepo (CSV has 386 rows) — count endpoint not supported
✓ sampling_event: exists in DBRepo (CSV has 1935 rows) — count endpoint not supported
✓ water_quality_measurement: exists in DBRepo (CSV has 1935 rows) — count endpoint not supported


In [10]:
print("=== VIEW VERIFICATION ===\n")

BASE_URL = "https://test.dbrepo.tuwien.ac.at"
views = client.get_views(database_id=DATABASE_ID)

for v in views:
    endpoint = f"{BASE_URL}/api/v1/database/{DATABASE_ID}/view/{v.id}/data"
    r = requests.get(
        endpoint,
        auth=(username, password),
        params={"size": 5, "page": 0},
        headers={"Accept": "application/json"}
    )
    
    if r.status_code == 200:
        data = r.json()
        if isinstance(data, list):
            rows = data
        else:
            rows = data.get("content", data.get("data", []))
        print(f"✓ {v.name}: returned {len(rows)} sample rows")
    else:
        print(f"⚠ {v.name}: status {r.status_code} — {r.text[:100]}")

=== VIEW VERIFICATION ===

✓ nutrient_pollution_features: returned 5 sample rows
✓ heavy_metal_pollution_features: returned 5 sample rows
✓ eutrophication_risk_indicators: returned 5 sample rows
✓ core_water_quality_features: returned 5 sample rows
